**Versión corregida 19.2.0 — 21 de noviembre de 2026.** Abre esta versión y ejecuta desde el inicio.

# 19 · Cierre experimental: cultivares balanceados, segundo cultivar y distancia visual

Este notebook **todavía no representa resultados ejecutados**. Ejecuta las sensibilidades solicitadas en R1.1c, R1.4 y R3.3. La factibilidad, los tamaños y la ausencia de duplicados se comprobaron con la metadata real antes de preparar el notebook. No necesita repetir las 120 corridas del notebook 15.

1. Audita los 60 checkpoints históricos que ya existen en Drive (opcional, sin GPU).
2. Mide separabilidad de dominios con un ResNet-50 **preentrenado y congelado**, en P4 restringido a las mismas cinco clases y P5. Escalador y clasificador usan entrenamiento; C se elige con validación; AUC se evalúa en test reservado. No es una estimación causal del efecto del cultivar ni del porcentaje explicado por edad.
3. **P6:** cuatro cultivares de OTHER × tres clases con exactamente 20 imágenes por celda; pares de ADT45 con la misma clase y edad. 480 imágenes, 144/48/48 por dominio. Cada cultivar aporta 12/4/4 por clase a train/val/test, de modo que cultivar no predice la clase por su frecuencia.
4. **P7:** AndraPonni frente al resto, con AndraPonni excluido del entrenamiento REST. Tres clases y 50 pares por clase, 300 imágenes, 90/30/30 por dominio. Ambas direcciones incluyen control entrenado en el propio dominio y transferencia evaluados en las mismas imágenes.

Clases: blast, dead_heart e hispa. Son sensibilidades pequeñas de tres clases, no réplicas equivalentes del P5 de cinco clases ni validación externa. Ambos diseños emparejan edad exactamente dentro de cada partición. Usan cuatro configuraciones, tres semillas y dos fuentes: **24 ajustes por diseño; 48 nuevos ajustes de 30 épocas en total, 96 evaluaciones**.

Las imágenes deben estar disponibles como `train_images/<clase>/<imagen>`. Si faltan imágenes, se ofrece restaurarlas de Kaggle con tu confirmación explícita dentro de Colab. Si las imágenes estaban en el runtime donde ejecutaste el notebook 15, puedes reutilizar esa ruta. También puede recuperar imágenes desde ZIP y cachés parciales; conserva los archivos verificados en Drive para próximas sesiones. La verificación SHA-256 impide entrenar con archivos distintos del pool auditado.

La duración depende de la GPU. Se guarda tiempo por corrida; la primera corrida permite estimar el resto. No se fija una estimación de unidades informáticas sin la tarifa vigente de tu sesión.

In [ ]:
%pip -q install timm==1.0.28 scipy pandas scikit-learn pillow kagglehub


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, json, base64, hashlib, io, zipfile
WORK = Path('/content/paddy19_code'); WORK.mkdir(exist_ok=True)


In [ ]:
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='84f658e9d92d75c2112924fc2e913a1d77b4f667021e33bcca2740fbde521b10'
with zipfile.ZipFile(io.BytesIO(payload)) as z: z.extractall(WORK/'inputs')


In [ ]:
(WORK/'paddy19_training_core.py').write_text('"""Paddy Doctor training core for P6/P7, version 19.1.0.\nFresh network fits; frozen class/age matching; source-validation model selection.\nNo claim of independent acquisition cohorts or external validation.\n"""\nfrom pathlib import Path\nimport os, sys, json, hashlib, time, random, gc, shutil, platform, itertools\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax, logsumexp\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix\n\nVERSION = \'19.2.0\'\nCLASSES = [\'blast\', \'dead_heart\', \'downy_mildew\', \'hispa\', \'normal\']\nDOMAINS = [\'ADT45\', \'OTHER\']\nMATCH_SEEDS = [31001, 31002, 31003, 31004, 31005]\nTRAIN_SEEDS = [2026, 2027, 2028]\nMODELS = {\n    \'resnet50\': \'resnet50.a1_in1k\',\n    \'convnext_tiny\': \'convnext_tiny.in12k_ft_in1k\',\n    \'vit_small_patch16_224\': \'vit_small_patch16_224.augreg_in21k_ft_in1k\',\n    \'swin_tiny_patch4_window7_224\': \'swin_tiny_patch4_window7_224.ms_in1k\',\n}\nRECIPE = dict(epochs=30, image_size=224, resize_size=256, batch_size=32,\n              lr=3e-4, weight_decay=.05, warmup_epochs=2,\n              warmup_start_factor=.1, cosine_eta_min=3e-6,\n              crop_scale=[.85, 1.], crop_ratio=[.75, 4/3], hflip=.5,\n              rotation_degrees=10, jitter=[.1, .1, .1, .02],\n              mean=[.485, .456, .406], std=[.229, .224, .225],\n              early_stopping=False, selection=\'source_validation_macro_f1\',\n              improvement_tolerance=1e-6, amp=\'fp16_on_cuda\',\n              loader_randomness=\'deterministic per-epoch seed; workers recreated each epoch\',\n              resume=\'completed runs skipped; interrupted run restarts from pretrained initialization\')\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_bytes(path, payload):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    with open(tmp, \'wb\') as f:\n        f.write(payload); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef write_json(path, obj):\n    atomic_bytes(path, (json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, default=str)+\'\\n\').encode())\n\n\ndef write_csv(path, df):\n    atomic_bytes(path, df.to_csv(index=False, lineterminator=\'\\n\').encode())\n\n\ndef freeze_json(path, obj):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != obj:\n            raise RuntimeError(f\'Cambió la configuración o el entorno: {path}. No mezclar resultados. Usa otro EXPERIMENT_NAME para un experimento diferente.\')\n    else: write_json(path, obj)\n\n\ndef load_metadata(path, expected_n=10329):\n    d = pd.read_csv(path, dtype={\'image_id\':str, \'label\':str, \'variety\':str, \'sha256\':str})\n    required={\'image_id\',\'label\',\'variety\',\'age\',\'sha256\'}\n    if not required.issubset(d): raise ValueError(f\'Metadata sin columnas: {required-set(d)}\')\n    if expected_n is not None and len(d)!=expected_n: raise ValueError(f\'Se esperaban {expected_n} imágenes, hay {len(d)}\')\n    if d[list(required)].isna().any().any() or d.image_id.duplicated().any(): raise ValueError(\'Metadata incompleta o IDs repetidos\')\n    if d.sha256.duplicated().any(): raise ValueError(\'Hay duplicados exactos en el pool near-clean\')\n    age=pd.to_numeric(d.age, errors=\'raise\')\n    if not np.isfinite(age).all() or (age!=np.floor(age)).any(): raise ValueError(\'Edad debe ser entera\')\n    d[\'age\']=age.astype(int)\n    if any(Path(x).name!=x for x in d.image_id): raise ValueError(\'image_id debe ser solo un nombre de archivo\')\n    d[\'domain\']=np.where(d.variety.eq(\'ADT45\'),\'ADT45\',\'OTHER\')\n    return d.sort_values(\'image_id\').reset_index(drop=True)\n\n\ndef construct_cohort(meta, match_seed):\n    """New pairing without replacement within cohort; class-stratified pair split 60/20/20."""\n    rng=np.random.default_rng(match_seed); records=[]\n    support=meta.groupby([\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).reindex(columns=DOMAINS,fill_value=0)\n    eligible=support.min(axis=1).groupby(level=0).sum()\n    if sorted(eligible[eligible>=100].index)!=CLASSES: raise ValueError(\'Las clases elegibles no coinciden con el P5 congelado\')\n    for label in CLASSES:\n        candidates=[]\n        sub=meta[meta.label.eq(label)]\n        for age in sorted(sub.age.unique()):\n            pools=[sub[sub.age.eq(age)&sub.domain.eq(domain)].sort_values(\'image_id\') for domain in DOMAINS]\n            pools=[p.iloc[rng.permutation(len(p))].reset_index(drop=True) for p in pools]\n            for i in range(min(map(len,pools))):\n                candidates.append(dict(label=label, age=int(age), ADT45_image_id=pools[0].iloc[i].image_id,\n                                       OTHER_image_id=pools[1].iloc[i].image_id))\n        chosen=[candidates[i] for i in rng.choice(len(candidates),100,replace=False)]\n        chosen=sorted(chosen,key=lambda r:(r[\'age\'],r[\'ADT45_image_id\'],r[\'OTHER_image_id\']))\n        order=rng.permutation(100); split=np.empty(100,dtype=object)\n        split[order[:60]]=\'train\';split[order[60:80]]=\'validation\';split[order[80:]]=\'test\'\n        for i,r in enumerate(chosen):\n            r.update(matched_pair_id=f\'M{match_seed}_{label}_{i:03d}\',pair_split=split[i],match_seed=int(match_seed))\n            records.append(r)\n    pairs=pd.DataFrame(records);lookup=meta.set_index(\'image_id\');rows=[]\n    for r in records:\n        for domain in DOMAINS:\n            image_id=r[f\'{domain}_image_id\'];m=lookup.loc[image_id]\n            rows.append(dict(image_id=image_id,label=r[\'label\'],age=r[\'age\'],variety=m.variety,\n                             domain=domain,sha256=m.sha256,matched_pair_id=r[\'matched_pair_id\'],\n                             pair_split=r[\'pair_split\'],match_seed=int(match_seed),\n                             visual_group_id=m.get(\'visual_group_id\',None)))\n    long=pd.DataFrame(rows).sort_values([\'domain\',\'pair_split\',\'label\',\'image_id\']).reset_index(drop=True)\n    validate_cohort(long)\n    return pairs,long\n\n\ndef validate_cohort(long):\n    if len(long)!=1000 or long.image_id.nunique()!=1000: raise ValueError(\'P5 debe tener 1000 imágenes únicas\')\n    if long.sha256.nunique()!=1000: raise ValueError(\'Duplicados SHA dentro de la cohorte\')\n    for split,n in [(\'train\',60),(\'validation\',20),(\'test\',20)]:\n        sub=long[long.pair_split.eq(split)]\n        counts=sub.groupby([\'domain\',\'label\']).size().reindex(pd.MultiIndex.from_product([DOMAINS,CLASSES]),fill_value=0)\n        if not counts.eq(n).all(): raise ValueError(\'Tamaño por clase/dominio incorrecto\')\n        age=sub.groupby([\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).reindex(columns=DOMAINS,fill_value=0)\n        if not age.ADT45.eq(age.OTHER).all(): raise ValueError(\'Edad no coincide exactamente entre dominios\')\n    if long.groupby(\'matched_pair_id\').size().ne(2).any(): raise ValueError(\'Cada par debe contener dos imágenes\')\n    for col in [\'pair_split\',\'label\',\'age\']:\n        if long.groupby(\'matched_pair_id\')[col].nunique().ne(1).any(): raise ValueError(f\'Par incoherente: {col}\')\n    if \'visual_group_id\' in long:\n        grouped=long.dropna(subset=[\'visual_group_id\']).groupby(\'visual_group_id\').pair_split.nunique()\n        if grouped.gt(1).any(): raise ValueError(\'Grupo visual cruza particiones: revisar pool near-clean\')\n\n\ndef prepare_design(meta, output, match_seeds=MATCH_SEEDS, reference=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True);cohorts={};rows=[]\n    for seed in match_seeds:\n        pairs,long=construct_cohort(meta,seed);cohorts[seed]=long\n        folder=output/f\'match_{seed}\';folder.mkdir(exist_ok=True)\n        for name,df in [(\'pairs.csv\',pairs),(\'assignments.csv\',long)]:\n            payload=df.to_csv(index=False,lineterminator=\'\\n\').encode();p=folder/name\n            if p.exists() and p.read_bytes()!=payload: raise RuntimeError(f\'Asignaciones congeladas diferentes: {p}\')\n            if not p.exists():atomic_bytes(p,payload)\n        rows.append(dict(match_seed=seed,n_images=1000,n_pairs=500,assignments_sha256=sha(folder/\'assignments.csv\')))\n    comparison=dict(cohorts)\n    if reference is not None:comparison[\'original_2026\']=reference\n    overlap=[]\n    for (a,da),(b,db) in itertools.combinations(comparison.items(),2):\n        for split in [\'all\',\'train\',\'validation\',\'test\']:\n            aa=set(da.image_id if split==\'all\' else da.loc[da.pair_split.eq(split),\'image_id\'])\n            bb=set(db.image_id if split==\'all\' else db.loc[db.pair_split.eq(split),\'image_id\'])\n            overlap.append(dict(cohort_a=a,cohort_b=b,partition=split,shared_images=len(aa&bb),union_images=len(aa|bb),jaccard=len(aa&bb)/len(aa|bb)))\n        for sa,sb in [(\'train\',\'test\'),(\'test\',\'train\')]:\n            aa=set(da.loc[da.pair_split.eq(sa),\'image_id\']);bb=set(db.loc[db.pair_split.eq(sb),\'image_id\'])\n            overlap.append(dict(cohort_a=a,cohort_b=b,partition=f\'{sa}_vs_{sb}\',shared_images=len(aa&bb),union_images=len(aa|bb),jaccard=len(aa&bb)/len(aa|bb)))\n    write_csv(output/\'overlap_between_cohorts.csv\',pd.DataFrame(overlap))\n    write_csv(output/\'design_manifest.csv\',pd.DataFrame(rows))\n    write_csv(output/\'candidate_support.csv\',meta.groupby([\'label\',\'age\',\'domain\']).size().rename(\'n\').reset_index())\n    return cohorts\n\n\ndef calibration_errors(y,p,bins=15,adaptive=False):\n    confidence=p.max(1); correct=(p.argmax(1)==y).astype(float)\n    edges=np.unique(np.quantile(confidence,np.linspace(0,1,bins+1))) if adaptive else np.linspace(0,1,bins+1)\n    if len(edges)<2:return float(abs(correct.mean()-confidence.mean())),float(abs(correct.mean()-confidence.mean()))\n    errors=[];weights=[]\n    for j in range(len(edges)-1):\n        use=(confidence>=edges[j] if j==0 else confidence>edges[j]) & (confidence<=edges[j+1])\n        if use.any():errors.append(abs(correct[use].mean()-confidence[use].mean()));weights.append(use.mean())\n    return float(np.dot(errors,weights)),float(max(errors,default=0.))\n\n\ndef score(y,logits,temperature=1.):\n    p=softmax(logits/temperature,axis=1);pred=p.argmax(1)\n    ece,mce=calibration_errors(y,p);adaptive,_=calibration_errors(y,p,10,True)\n    return dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),\n                macro_f1=float(f1_score(y,pred,labels=range(len(CLASSES)),average=\'macro\',zero_division=0)),\n                nll=float(np.mean(logsumexp(logits/temperature,axis=1)-(logits/temperature)[np.arange(len(y)),y])),\n                brier=float(np.mean(np.sum((p-np.eye(len(CLASSES))[y])**2,axis=1))),\n                ece_15=ece,adaptive_ece_10=adaptive,mce_15=mce)\n\n\ndef fit_temperature(y,logits):\n    result=minimize_scalar(lambda u:score(y,logits,np.exp(u))[\'nll\'],method=\'bounded\',\n                           bounds=(np.log(.05),np.log(20)),options={\'xatol\':1e-5})\n    if not result.success or not np.isfinite(result.fun):raise RuntimeError(\'Falló el ajuste de temperatura fuente\')\n    return float(np.exp(result.x)),dict(success=True,log_temperature=float(result.x),source_nll=float(result.fun),nfev=int(result.nfev))\n\n\ndef load_torch():\n    global torch,timm,T,InterpolationMode,Image,DataLoader\n    import torch,timm\n    from torchvision import transforms as T\n    from torchvision.transforms import InterpolationMode\n    from PIL import Image\n    from torch.utils.data import DataLoader\n\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n\n\ndef state_hash(state):\n    h=hashlib.sha256()\n    for name,t in sorted(state.items()):\n        t=t.detach().cpu().contiguous()\n        h.update(name.encode());h.update(str(t.dtype).encode());h.update(str(tuple(t.shape)).encode())\n        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())\n    return h.hexdigest()\n\n\ndef atomic_torch(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\');torch.save(obj,tmp);os.replace(tmp,path)\n\n\ndef environment():\n    import importlib.metadata\n    return dict(python=platform.python_version(),packages={x:importlib.metadata.version(x) for x in\n                [\'torch\',\'torchvision\',\'timm\',\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'Pillow\']},\n                cuda=str(torch.version.cuda),cudnn=int(torch.backends.cudnn.version() or 0),\n                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\')\n\n\ndef ensure_pretrained(cache,models=MODELS):\n    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)\n    for key,tag in models.items():\n        p=cache/(key+\'.pt\');manifest=cache/(key+\'.json\')\n        if p.exists() and manifest.exists():\n            info=json.loads(manifest.read_text())\n            if info[\'tag\']!=tag or sha(p)!=info[\'file_sha256\']:raise RuntimeError(\'Caché de pesos inconsistente\')\n            print(\'Pesos verificados:\',tag);continue\n        print(\'Descargando una vez:\',tag,flush=True)\n        seed_all(0)\n        model=timm.create_model(tag,pretrained=True,num_classes=1000).cpu()\n        state={k:v.detach().cpu() for k,v in model.state_dict().items()}\n        cfg=json.loads(json.dumps(model.pretrained_cfg,default=str))\n        atomic_torch(p,state)\n        write_json(manifest,dict(tag=tag,file_sha256=sha(p),tensor_sha256=state_hash(state),pretrained_cfg=cfg))\n        del model,state;gc.collect()\n\n\ndef transforms(training):\n    if training:\n        return T.Compose([T.Resize(256,interpolation=InterpolationMode.BICUBIC),\n            T.RandomResizedCrop(224,scale=(.85,1.),ratio=(.75,4/3),interpolation=InterpolationMode.BICUBIC),\n            T.RandomHorizontalFlip(.5),T.RandomRotation(10,interpolation=InterpolationMode.BILINEAR),\n            T.ColorJitter(brightness=.1,contrast=.1,saturation=.1,hue=.02),T.ToTensor(),T.Normalize(RECIPE[\'mean\'],RECIPE[\'std\'])])\n    return T.Compose([T.Resize(256,interpolation=InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize(RECIPE[\'mean\'],RECIPE[\'std\'])])\n\n\nclass ImageDataset:\n    def __init__(self,df,lookup,training=False):\n        self.df=df.reset_index(drop=True);self.lookup=lookup;self.transform=transforms(training)\n    def __len__(self):return len(self.df)\n    def __getitem__(self,i):\n        r=self.df.iloc[i]\n        with Image.open(self.lookup[r.image_id]) as im:x=self.transform(im.convert(\'RGB\'))\n        return x,CLASSES.index(r.label)\n\n\ndef worker_seed(worker_id):\n    seed=torch.initial_seed()%(2**32);np.random.seed(seed);random.seed(seed)\n\n\ndef loader(df,lookup,seed,training,workers=2):\n    generator=torch.Generator().manual_seed(seed)\n    return DataLoader(ImageDataset(df,lookup,training),batch_size=RECIPE[\'batch_size\'],shuffle=training,\n                      generator=generator,num_workers=workers,pin_memory=torch.cuda.is_available(),drop_last=False,\n                      persistent_workers=False,worker_init_fn=worker_seed)\n\n\ndef predict(model,data_loader,device):\n    model.eval();ys=[];zs=[]\n    with torch.no_grad():\n        for x,y in data_loader:\n            with torch.amp.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                z=model(x.to(device,non_blocking=True))\n            ys.append(y.numpy());zs.append(z.float().cpu().numpy().astype(np.float64))\n    return np.concatenate(ys),np.concatenate(zs)\n\n\ndef fit(model,train_df,val_df,lookup,seed,device,history_path,workers=2,epochs=None):\n    epochs=epochs or RECIPE[\'epochs\'];optimizer=torch.optim.AdamW(model.parameters(),lr=RECIPE[\'lr\'],weight_decay=RECIPE[\'weight_decay\'])\n    warm=torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=.1,end_factor=1.,total_iters=2)\n    cosine=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,epochs-2),eta_min=3e-6)\n    scheduler=torch.optim.lr_scheduler.SequentialLR(optimizer,[warm,cosine],milestones=[2])\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\');criterion=torch.nn.CrossEntropyLoss()\n    best=-np.inf;best_epoch=None;best_state=None;history=[]\n    val_loader=loader(val_df,lookup,seed,False,workers)\n    for epoch in range(1,epochs+1):\n        epoch_seed=(int(seed)*100003+epoch)%(2**32)\n        seed_all(epoch_seed)\n        train_loader=loader(train_df,lookup,epoch_seed,True,workers)\n        start=time.perf_counter();model.train();loss_sum=0.;n=0\n        for x,y in train_loader:\n            x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True);optimizer.zero_grad(set_to_none=True)\n            with torch.amp.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n                logits=model(x);loss=criterion(logits,y)\n            if not torch.isfinite(loss):raise RuntimeError(\'Loss no finita; se detiene sin marcar COMPLETE\')\n            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n            loss_sum+=float(loss.detach())*len(y);n+=len(y)\n        yv,zv=predict(model,val_loader,device);metrics=score(yv,zv)\n        if not np.isfinite(zv).all():raise RuntimeError(\'Logits de validación no finitos\')\n        if metrics[\'macro_f1\']>best+RECIPE[\'improvement_tolerance\']:\n            best=metrics[\'macro_f1\'];best_epoch=epoch\n            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        history.append(dict(epoch=epoch,train_loss=loss_sum/n,val_loss=metrics[\'nll\'],val_macro_f1=metrics[\'macro_f1\'],\n                            val_accuracy=metrics[\'accuracy\'],learning_rate=optimizer.param_groups[0][\'lr\'],epoch_seconds=time.perf_counter()-start))\n        write_csv(history_path,pd.DataFrame(history));scheduler.step()\n        if epoch==1 or epoch%5==0:print(f\'  época {epoch:02}/{epochs}: val F1={metrics["macro_f1"]:.4f}; mejor={best:.4f}\',flush=True)\n    model.load_state_dict(best_state)\n    return dict(best_epoch=best_epoch,best_val_macro_f1=best,epochs_completed=epochs),best_state\n\n\ndef export_predictions(model,df,lookup,seed,device,destination,temperature,workers=2):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    y,z=predict(model,loader(df,lookup,seed,False,workers),device)\n    if not np.isfinite(z).all():raise RuntimeError(\'Logits no finitos\')\n    raw=softmax(z,axis=1);scaled=softmax(z/temperature,axis=1)\n    data=df.reset_index(drop=True).copy();data[\'y_true_idx\']=y\n    for name,p in [(\'raw\',raw),(\'TS\',scaled)]:\n        data[f\'y_pred_{name}\']=[CLASSES[i] for i in p.argmax(1)];data[f\'confidence_{name}\']=p.max(1)\n        for i,label in enumerate(CLASSES):data[f\'prob_{name}__{label}\']=p[:,i]\n    for i,label in enumerate(CLASSES):data[f\'logit__{label}\']=z[:,i]\n    write_csv(destination/\'predictions.csv\',data)\n    np.savez_compressed(destination/\'logits.npz\',y_true=y,logits=z,image_id=df.image_id.to_numpy(dtype=str),class_names=np.array(CLASSES))\n    precision,recall,f1,support=precision_recall_fscore_support(y,raw.argmax(1),labels=range(len(CLASSES)),zero_division=0)\n    write_csv(destination/\'per_class.csv\',pd.DataFrame(dict(class_name=CLASSES,precision=precision,recall=recall,f1=f1,support=support)))\n    cm=pd.DataFrame(confusion_matrix(y,raw.argmax(1),labels=range(len(CLASSES))),columns=CLASSES);cm.insert(0,\'true_class\',CLASSES)\n    write_csv(destination/\'confusion_counts.csv\',cm)\n    metrics=dict(raw=score(y,z),TS=score(y,z,temperature));write_json(destination/\'metrics.json\',metrics)\n    return metrics\n\n\ndef prepare_images(meta,cohorts,image_root,cache_root):\n    from PIL import Image\n    image_root=Path(image_root);cache_root=Path(cache_root);cache_root.mkdir(parents=True,exist_ok=True)\n    selected=set().union(*(set(d.image_id) for d in cohorts.values()))\n    lookup={};metadata=meta.set_index(\'image_id\');rows=[]\n    for i,image_id in enumerate(sorted(selected),1):\n        row=metadata.loc[image_id];source=image_root/row.label/image_id;dest=cache_root/row.label/image_id\n        if not dest.exists():\n            if not source.is_file():raise FileNotFoundError(f\'No se encuentra {source}\')\n            dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,dest)\n        digest=sha(dest)\n        if digest!=row.sha256:raise ValueError(f\'Los bytes de {image_id} no coinciden con el pool auditado. No se entrena.\')\n        with Image.open(dest) as image:image.verify()\n        lookup[image_id]=str(dest);rows.append(dict(image_id=image_id,sha256=digest))\n        if i%500==0:print(f\'  imágenes verificadas: {i}/{len(selected)}\',flush=True)\n    return lookup,pd.DataFrame(rows)\n\n\ndef complete_ok(run_dir,signature,model_dir):\n    marker=Path(run_dir)/\'COMPLETE.json\'\n    if not marker.exists():return False\n    payload=json.loads(marker.read_text())\n    if payload[\'signature\']!=signature:raise RuntimeError(\'COMPLETE corresponde a otra configuración\')\n    for item in payload[\'files\']:\n        base=Path(model_dir) if item[\'area\']==\'models\' else Path(run_dir)\n        p=base/item[\'path\']\n        if not p.is_file() or sha(p)!=item[\'sha256\']:raise RuntimeError(f\'Corrida marcada completa pero archivo ausente/cambiado: {p}\')\n    return True\n\n\ndef run_one(cohort,match_seed,key,train_seed,train_domain,lookup,root,model_root,weight_cache,signature,workers=2,device=None):\n    run_name=f\'match_{match_seed}/train_{train_domain}__{key}__seed_{train_seed}\'\n    run_dir=Path(root)/\'runs\'/run_name;model_dir=Path(model_root)/run_name\n    if complete_ok(run_dir,signature,model_dir):print(\'Completado, se omite:\',run_name,flush=True);return\n    run_dir.mkdir(parents=True,exist_ok=True);model_dir.mkdir(parents=True,exist_ok=True)\n    print(\'\\nENTRENANDO\',run_name,flush=True);seed_all(train_seed)\n    start=time.perf_counter();device=torch.device(device or \'cuda\')\n    model=timm.create_model(MODELS[key],pretrained=False,num_classes=1000)\n    model.load_state_dict(torch.load(Path(weight_cache)/(key+\'.pt\'),map_location=\'cpu\',weights_only=True))\n    model.reset_classifier(len(CLASSES))\n    initialization_hash=state_hash(model.state_dict())\n    model=model.to(device)\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n    train=cohort[cohort.domain.eq(train_domain)&cohort.pair_split.eq(\'train\')].reset_index(drop=True)\n    val=cohort[cohort.domain.eq(train_domain)&cohort.pair_split.eq(\'validation\')].reset_index(drop=True)\n    provenance=dict(signature=signature,match_seed=match_seed,model_key=key,tag=MODELS[key],train_seed=train_seed,\n        train_domain=train_domain,initialized_tensor_sha256=initialization_hash,\n        pretrained=json.loads((Path(weight_cache)/(key+\'.json\')).read_text()),\n        n_parameters=sum(p.numel() for p in model.parameters()),recipe=RECIPE)\n    write_json(run_dir/\'provenance.json\',provenance)\n    best,best_state=fit(model,train,val,lookup,train_seed,device,run_dir/\'history.csv\',workers)\n    checkpoint=model_dir/\'best_model.pt\'\n    atomic_torch(checkpoint,dict(model_state_dict=best_state,provenance=provenance,best=best,class_names=CLASSES))\n    yv,zv=predict(model,loader(val,lookup,train_seed,False,workers),device)\n    temperature,diagnostics=fit_temperature(yv,zv)\n    write_json(run_dir/\'temperature.json\',dict(temperature=temperature,fit_domain=train_domain,diagnostics=diagnostics,\n               note=\'source validation reused for checkpoint and temperature; no formal calibration guarantee\'))\n    export_predictions(model,val,lookup,train_seed,device,run_dir/\'source_validation\',temperature,workers)\n    evaluations=[]\n    for target in DOMAINS:\n        test=cohort[cohort.domain.eq(target)&cohort.pair_split.eq(\'test\')].reset_index(drop=True)\n        metrics=export_predictions(model,test,lookup,train_seed,device,run_dir/f\'eval__{target}\',temperature,workers)\n        evaluations.append(dict(match_seed=match_seed,model_key=key,train_seed=train_seed,train_domain=train_domain,\n                                eval_domain=target,**{f\'{cal}_{name}\':value for cal,m in metrics.items() for name,value in m.items()}))\n    write_csv(run_dir/\'evaluations.csv\',pd.DataFrame(evaluations))\n    write_json(run_dir/\'training_summary.json\',dict(**best,wall_seconds=time.perf_counter()-start,peak_gpu_memory_gb=torch.cuda.max_memory_allocated()/1024**3 if device.type==\'cuda\' else 0.))\n    files=[]\n    for area,folder in [(\'results\',run_dir),(\'models\',model_dir)]:\n        for p in sorted(folder.rglob(\'*\')):\n            if p.is_file() and p.name!=\'COMPLETE.json\' and not p.name.endswith(\'.tmp\'):\n                files.append(dict(area=area,path=p.relative_to(folder).as_posix(),sha256=sha(p)))\n    write_json(run_dir/\'COMPLETE.json\',dict(signature=signature,files=files))\n    del model,best_state;gc.collect();torch.cuda.empty_cache()\n\n\ndef summarize(root,expected_runs=120):\n    root=Path(root);parts=[]\n    for marker in sorted((root/\'runs\').rglob(\'COMPLETE.json\')):parts.append(pd.read_csv(marker.parent/\'evaluations.csv\'))\n    if not parts:raise RuntimeError(\'Todavía no hay corridas completas\')\n    evaluations=pd.concat(parts,ignore_index=True);write_csv(root/\'evaluations_all.csv\',evaluations)\n    rows=[]\n    for (match,key,seed,target),g in evaluations.groupby([\'match_seed\',\'model_key\',\'train_seed\',\'eval_domain\']):\n        control=g[g.train_domain.eq(target)];transfer=g[~g.train_domain.eq(target)]\n        if len(control)!=1 or len(transfer)!=1:continue\n        rows.append(dict(match_seed=match,model_key=key,train_seed=seed,target=target,\n                         control_f1=float(control.raw_macro_f1.iloc[0]),transfer_f1=float(transfer.raw_macro_f1.iloc[0]),\n                         loss=float(control.raw_macro_f1.iloc[0]-transfer.raw_macro_f1.iloc[0])))\n    losses=pd.DataFrame(rows)\n    if losses.empty:raise RuntimeError(\'Aún falta el dominio pareado para comparar el mismo objetivo\')\n    write_csv(root/\'fixed_target_losses_by_seed.csv\',losses)\n    complete=(losses.groupby([\'match_seed\',\'model_key\',\'target\']).train_seed.nunique()==3)\n    cohort=losses.groupby([\'match_seed\',\'model_key\',\'target\']).agg(n_seeds=(\'train_seed\',\'nunique\'),\n        loss_mean=(\'loss\',\'mean\'),loss_sd_seeds=(\'loss\',\'std\'),loss_min_seed=(\'loss\',\'min\'),loss_max_seed=(\'loss\',\'max\'),\n        control_f1=(\'control_f1\',\'mean\'),transfer_f1=(\'transfer_f1\',\'mean\')).reset_index()\n    write_csv(root/\'fixed_target_losses_by_cohort.csv\',cohort)\n    full=cohort[cohort.n_seeds.eq(3)]\n    across=full.groupby([\'model_key\',\'target\']).agg(n_rematchings=(\'match_seed\',\'nunique\'),loss_mean=(\'loss_mean\',\'mean\'),\n        loss_sd_between_rematchings=(\'loss_mean\',\'std\'),loss_min=(\'loss_mean\',\'min\'),loss_max=(\'loss_mean\',\'max\'),\n        positive_losses=(\'loss_mean\',lambda s:int((s>0).sum()))).reset_index()\n    write_csv(root/\'rematching_sensitivity_summary.csv\',across)\n    n=len(parts);write_json(root/\'completion_summary.json\',dict(completed_runs=n,expected_runs=expected_runs,\n        evaluations=len(evaluations),complete=n==expected_runs,statistical_note=\'overlapping repeated constructions; no independent-cohort p-value or external-validation claim\'))\n    print(f\'Corridas completas: {n}/{expected_runs}; evaluaciones: {len(evaluations)}/{expected_runs*2}\')\n    return losses,cohort,across\n\n',encoding='utf-8')
sys.path.insert(0,str(WORK))


In [ ]:
(WORK/'paddy19_cultivar_extensions.py').write_text('"""Two additional cultivar designs and frozen-pretrained domain discrimination.\nNo Kaggle authentication or credential discovery. Images must already be local.\nThe 3-class experiments are sensitivities, not replacements for the primary P5.\n"""\nfrom pathlib import Path\nimport json,hashlib,sys,itertools,time\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import linprog\nfrom sklearn.metrics import roc_auc_score,balanced_accuracy_score,log_loss,f1_score\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\n\nCLASSES=[\'blast\',\'dead_heart\',\'hispa\']\nSEEDS=[2026,2027,2028]\nBALANCED_VARIETIES=[\'AndraPonni\',\'KarnatakaPonni\',\'Onthanel\',\'Ponni\']\n\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef write(p,obj):Path(p).parent.mkdir(parents=True,exist_ok=True);Path(p).write_text(json.dumps(obj,indent=2,default=str)+\'\\n\')\n\ndef pair_design(meta,protocol):\n    """Freeze allocation from metadata alone, before any model outcomes."""\n    rng=np.random.default_rng(19001 if protocol==\'P6\' else 19002);rows=[]\n    a=meta[meta.variety.eq(\'ADT45\')].copy() if protocol==\'P6\' else meta[meta.variety.eq(\'AndraPonni\')].copy()\n    b=meta[meta.variety.isin(BALANCED_VARIETIES)].copy() if protocol==\'P6\' else meta[~meta.variety.eq(\'AndraPonni\')].copy()\n    domains=[\'ADT45\',\'OTHER_balanced\'] if protocol==\'P6\' else [\'AndraPonni\',\'REST\']\n    for label in CLASSES:\n        aa=a[a.label.eq(label)];bb=b[b.label.eq(label)];pairs=[]\n        if protocol==\'P6\':\n            ages=sorted(set(aa.age)&set(bb.age));variables=[(v,age) for v in BALANCED_VARIETIES for age in ages]\n            bounds=[(0,int(((bb.variety==v)&(bb.age==age)).sum())) for v,age in variables]\n            eq=np.array([[int(v==vv) for v,age in variables] for vv in BALANCED_VARIETIES]);ub=np.array([[int(age==ag) for v,age in variables] for ag in ages])\n            result=linprog(rng.random(len(variables)),A_ub=ub,b_ub=[int((aa.age==ag).sum()) for ag in ages],A_eq=eq,b_eq=[20]*4,bounds=bounds,method=\'highs\')\n            if not result.success:raise ValueError(f\'P6 exact-age allocation infeasible for {label}: {result.message}\')\n            allocation=np.rint(result.x).astype(int)\n            if not np.allclose(allocation,result.x,atol=1e-7):raise ValueError(\'Non-integer network allocation\')\n            pools={age:list(rng.permutation(aa.loc[aa.age.eq(age),\'image_id\'].to_numpy())) for age in ages}\n            for (v,age),n in zip(variables,allocation):\n                chosen=rng.choice(bb.loc[bb.variety.eq(v)&bb.age.eq(age),\'image_id\'],n,replace=False)\n                for iid in chosen:pairs.append((pools[age].pop(),iid,int(age),v))\n            # Equal cultivar x class counts in every partition; pairs stay together.\n            split=[]\n            for v in BALANCED_VARIETIES:\n                ids=[i for i,p in enumerate(pairs) if p[3]==v];assert len(ids)==20\n                for j,i in enumerate(rng.permutation(ids)):split.append((i,\'train\' if j<12 else \'validation\' if j<16 else \'test\'))\n            splits=dict(split)\n        else:\n            for age in sorted(set(aa.age)&set(bb.age)):\n                pa=rng.permutation(aa.loc[aa.age.eq(age),\'image_id\']);pb=rng.permutation(bb.loc[bb.age.eq(age),\'image_id\'])\n                pairs.extend((x,y,int(age),None) for x,y in zip(pa,pb))\n            if len(pairs)<50:raise ValueError(f\'P7 requires 50 exact-age pairs/class; {label} has {len(pairs)}\')\n            pairs=[pairs[i] for i in rng.choice(len(pairs),50,replace=False)]\n            splits={i:(\'train\' if j<30 else \'validation\' if j<40 else \'test\') for j,i in enumerate(rng.permutation(50))}\n        for i,(ia,ib,age,variety) in enumerate(pairs):\n            for domain,iid in zip(domains,[ia,ib]):rows.append(dict(image_id=iid,label=label,age=age,domain=domain,pair_split=splits[i],matched_pair_id=f\'{protocol}_{label}_{i:03d}\',match_seed=19001 if protocol==\'P6\' else 19002))\n    d=pd.DataFrame(rows).merge(meta.drop(columns=[\'domain\'],errors=\'ignore\'),on=[\'image_id\',\'label\',\'age\'],validate=\'one_to_one\')\n    assert d.image_id.nunique()==len(d) and d.sha256.nunique()==len(d)\n    assert (d.groupby(\'matched_pair_id\').pair_split.nunique()==1).all()\n    assert (d.groupby([\'pair_split\',\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).nunique(axis=1)==1).all()\n    if d.dropna(subset=[\'visual_group_id\']).groupby(\'visual_group_id\').pair_split.nunique().gt(1).any():raise ValueError(\'Visual group crosses partitions\')\n    if protocol==\'P6\':\n        c=d[d.domain.eq(domains[1])].groupby([\'pair_split\',\'label\',\'variety\']).size().unstack(fill_value=0)\n        assert (c.nunique(axis=1)==1).all() and list(c.columns)==sorted(BALANCED_VARIETIES)\n    else:assert d.loc[d.domain.eq(\'REST\'),\'variety\'].ne(\'AndraPonni\').all()\n    return d.sort_values([\'domain\',\'pair_split\',\'label\',\'image_id\']).reset_index(drop=True),domains\n\ndef prepare(meta_path,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);meta=pd.read_csv(meta_path);designs={};counts=[]\n    for protocol in [\'P6\',\'P7\']:\n        frozen=Path(meta_path).parent/\'frozen_designs\'/f\'{protocol}_assignments.csv\'\n        if frozen.exists():\n            manifest=json.loads((frozen.parent/\'SHA256.json\').read_text())\n            if digest(frozen)!=manifest[frozen.name]:raise ValueError(\'Asignaciones originales modificadas\')\n            payload=frozen.read_bytes();d=pd.read_csv(frozen)\n            domains=[\'ADT45\',\'OTHER_balanced\'] if protocol==\'P6\' else [\'AndraPonni\',\'REST\']\n            reference=meta.set_index(\'image_id\').loc[d.image_id].reset_index()\n            for col in [\'label\',\'age\',\'variety\',\'sha256\']:\n                if not (reference[col].to_numpy()==d[col].to_numpy()).all():raise ValueError(\'Asignaciones y metadata no coinciden: \'+col)\n            if d.image_id.duplicated().any() or d.sha256.duplicated().any():raise ValueError(\'Imágenes repetidas\')\n            if d.groupby(\'matched_pair_id\').size().ne(2).any():raise ValueError(\'Pareja incompleta\')\n            if d.groupby(\'matched_pair_id\').pair_split.nunique().ne(1).any():raise ValueError(\'Pareja en dos splits\')\n            if not (d.groupby([\'pair_split\',\'label\',\'age\',\'domain\']).size().unstack(fill_value=0).nunique(axis=1)==1).all():raise ValueError(\'Edades no emparejadas\')\n            if d.dropna(subset=[\'visual_group_id\']).groupby(\'visual_group_id\').pair_split.nunique().gt(1).any():raise ValueError(\'Grupo visual cruza particiones\')\n        else:\n            d,domains=pair_design(meta,protocol);payload=d.to_csv(index=False).encode()\n        p=out/f\'{protocol}_assignments.csv\'\n        if p.exists() and p.read_bytes()!=payload:\n            old=pd.read_csv(p)\n            try:pd.testing.assert_frame_equal(old,d,check_dtype=False)\n            except AssertionError as e:raise RuntimeError(\'Las asignaciones guardadas difieren; no se sobrescriben.\') from e\n        p.write_bytes(payload);designs[protocol]=(d,domains)\n        counts.append(dict(protocol=protocol,n_images=len(d),n_classes=3,domains=domains,assignments_sha256=digest(p),seed=int(d.match_seed.iloc[0])))\n        d.groupby([\'domain\',\'pair_split\',\'label\']).size().rename(\'n\').reset_index().to_csv(out/f\'{protocol}_counts.csv\',index=False)\n    write(out/\'design_manifest.json\',{\'metadata_sha256\':digest(meta_path),\'designs\':counts,\'selection\':\'metadata-only frozen design; three classes supported by both sensitivities\',\'no_external_validation\':True})\n    return meta,designs\n\ndef run_training(meta_path,out,image_root,model_root,weight_cache,workers=2):\n    import paddy19_training_core as core\n    core.load_torch()\n    if not core.torch.cuda.is_available():raise RuntimeError(\'Activa una GPU para los 48 entrenamientos nuevos.\')\n    if core.timm.__version__!=\'1.0.28\':raise RuntimeError(\'Se requiere timm 1.0.28\')\n    out=Path(out);meta,designs=prepare(meta_path,out/\'design\');core.CLASSES=CLASSES\n    # Verify available images before requesting any pretrained downloads.\n    lookup,receipt=core.prepare_images(meta,{k:d for k,(d,ds) in designs.items()},image_root,\'/content/paddy19_image_cache\')\n    receipt.to_csv(out/\'verified_training_images.csv\',index=False)\n    core.ensure_pretrained(weight_cache)\n    env=core.environment();core.freeze_json(out/\'environment_lock.json\',env)\n    tasks_done=0;elapsed_start=time.perf_counter()\n    for protocol,(d,domains) in designs.items():\n        core.DOMAINS=domains;root=out/protocol;root.mkdir(exist_ok=True)\n        plan=dict(protocol=protocol,classes=CLASSES,domains=domains,training_seeds=SEEDS,recipe=core.RECIPE,models=core.MODELS,environment=env,workers=workers,assignments_sha256=digest(out/\'design\'/f\'{protocol}_assignments.csv\'),source_sha256=digest(__file__),core_sha256=digest(core.__file__),pretrained={k:json.loads((Path(weight_cache)/(k+\'.json\')).read_text()) for k in core.MODELS})\n        core.freeze_json(root/\'experiment_plan.json\',plan);signature=hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()\n        for key,seed,domain in itertools.product(core.MODELS,SEEDS,domains):\n            print(f\'[{tasks_done+1}/48] {protocol} | {key} | seed={seed} | fuente={domain}\',flush=True)\n            core.run_one(d,int(d.match_seed.iloc[0]),key,seed,domain,lookup,root,Path(model_root)/protocol,weight_cache,signature,workers)\n            tasks_done+=1\n            print(f\'Avance: {tasks_done}/48; tiempo de esta sesión: {(time.perf_counter()-elapsed_start)/60:.1f} min\',flush=True)\n        core.summarize(root,expected_runs=24)\n    summarize(out)\n\ndef summarize(out,B=5000):\n    out=Path(out);results=[];intervals=[]\n    for protocol in [\'P6\',\'P7\']:\n        runs=[]\n        for marker in sorted((out/protocol/\'runs\').rglob(\'COMPLETE.json\')):\n            for p in sorted(marker.parent.glob(\'eval__*/predictions.csv\')):\n                d=pd.read_csv(p);prov=json.loads((marker.parent/\'provenance.json\').read_text());runs.append((prov,p.parent.name[6:],d.sort_values(\'image_id\').reset_index(drop=True)))\n        if len(runs)!=48:raise RuntimeError(f\'{protocol}: {len(runs)}/48 evaluations; not complete\')\n        for target,model in itertools.product(sorted({x[1] for x in runs}),sorted({x[0][\'model_key\'] for x in runs})):\n            selected=[r for r in runs if r[1]==target and r[0][\'model_key\']==model];y=selected[0][2].label.to_numpy();ids=selected[0][2].image_id.tolist();rng=np.random.default_rng(19100+(protocol==\'P7\'))\n            index=np.concatenate([rng.choice(np.flatnonzero(y==cl),size=(B,int((y==cl).sum())),replace=True) for cl in CLASSES],axis=1);rep=[];point=[]\n            for seed in SEEDS:\n                c=next(d for p,t,d in selected if p[\'train_domain\']==target and p[\'train_seed\']==seed);t=next(d for p,t,d in selected if p[\'train_domain\']!=target and p[\'train_seed\']==seed)\n                assert ids==c.image_id.tolist()==t.image_id.tolist()\n                cp=c.y_pred_raw.to_numpy();tp=t.y_pred_raw.to_numpy();cf=f1_score(y,cp,labels=CLASSES,average=\'macro\',zero_division=0);tf=f1_score(y,tp,labels=CLASSES,average=\'macro\',zero_division=0)\n                point.append(cf-tf);results.append(dict(protocol=protocol,target=target,model=model,seed=seed,control_f1=cf,transfer_f1=tf,loss=cf-tf,n_unique_test=len(y)))\n                def bootf(pred):\n                    yy=np.searchsorted(CLASSES,y)[index];pp=np.searchsorted(CLASSES,pred)[index];cm=np.bincount((yy*3+pp+np.arange(B)[:,None]*9).ravel(),minlength=B*9).reshape(B,3,3);den=cm.sum(1)+cm.sum(2);return np.divide(2*np.diagonal(cm,axis1=1,axis2=2),den,out=np.zeros((B,3)),where=den>0).mean(1)\n                rep.append(bootf(cp)-bootf(tp))\n            lo,hi=np.quantile(np.mean(rep,axis=0),[.025,.975]);intervals.append(dict(protocol=protocol,target=target,model=model,mean_seed_loss=np.mean(point),image_ci95_lower=lo,image_ci95_upper=hi,replicates=B,n_unique_test=len(y),scope=\'three fixed seeds; secondary 3-class sensitivity\'))\n    pd.DataFrame(results).to_csv(out/\'19_losses_by_seed.csv\',index=False);pd.DataFrame(intervals).to_csv(out/\'19_paired_image_intervals.csv\',index=False)\n\ndef domain_embeddings(meta_path,p4_path,p5_path,image_root,out,weight_cache,workers=2,B=5000):\n    """Frozen ResNet features; scaler/train C selection on train/validation only."""\n    import paddy19_training_core as core\n    core.load_torch();out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    meta=pd.read_csv(meta_path);p4=pd.read_csv(p4_path).rename(columns={\'domain_split\':\'pair_split\'});p5=pd.read_csv(p5_path)\n    if \'pair_split\' not in p5 and \'split\' in p5:p5=p5.rename(columns={\'split\':\'pair_split\'})\n    specs={\'P4_same5\':p4[p4.label.isin([\'blast\',\'dead_heart\',\'downy_mildew\',\'hispa\',\'normal\'])].copy(),\'P5\':p5.copy()}\n    selected=pd.concat([d for d in specs.values()]).drop_duplicates(\'image_id\')\n    lookup,receipt=core.prepare_images(meta,{\'embedding\':selected},image_root,\'/content/paddy19_image_cache\');receipt.to_csv(out/\'verified_embedding_images.csv\',index=False)\n    core.ensure_pretrained(weight_cache,models={\'resnet50\':core.MODELS[\'resnet50\']})\n    cfg=json.loads((Path(weight_cache)/\'resnet50.json\').read_text());signature=dict(metadata_sha256=digest(meta_path),p4_assignments=digest(p4_path),p5_assignments=digest(p5_path),pretrained=cfg,environment=core.environment(),source_sha256=digest(__file__),transform=\'resize256 center224 bicubic ImageNet norm\',tag=core.MODELS[\'resnet50\'])\n    core.freeze_json(out/\'embedding_plan.json\',signature)\n    cache=out/\'frozen_resnet_features.npz\';df=selected.sort_values(\'image_id\').reset_index(drop=True);core.CLASSES=sorted(meta.label.unique())\n    if cache.exists():\n        receipt_path=out/\'features_sha256.json\'\n        if not receipt_path.exists() or json.loads(receipt_path.read_text())[\'sha256\']!=digest(cache):raise RuntimeError(\'Caché de embeddings incompleta o modificada; retira frozen_resnet_features.npz y features_sha256.json para recalcularla.\')\n        a=np.load(cache,allow_pickle=False);assert a[\'image_ids\'].tolist()==df.image_id.tolist();features=a[\'features\']\n    else:\n        device=core.torch.device(\'cuda\' if core.torch.cuda.is_available() else \'cpu\');model=core.timm.create_model(core.MODELS[\'resnet50\'],pretrained=False,num_classes=1000)\n        model.load_state_dict(core.torch.load(Path(weight_cache)/\'resnet50.pt\',weights_only=True,map_location=\'cpu\'));model.reset_classifier(0);model.to(device).eval();chunks=[]\n        with core.torch.no_grad():\n            for x,y in core.loader(df,lookup,19000,False,workers):chunks.append(model(x.to(device)).float().cpu().numpy())\n        features=np.concatenate(chunks);tmp=cache.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,features=features,image_ids=df.image_id.to_numpy(dtype=str));tmp.replace(cache);write(out/\'features_sha256.json\',dict(sha256=digest(cache)));del model\n    if features.ndim!=2 or len(features)!=len(df) or not np.isfinite(features).all():raise ValueError(\'Embeddings inválidos\')\n    print(\'Embeddings completos:\',features.shape,flush=True)\n    fi={iid:i for i,iid in enumerate(df.image_id)};rows=[];predrows=[]\n    for protocol,d in specs.items():\n        print(\'Clasificador de dominio:\',protocol,flush=True)\n        d=d.sort_values(\'image_id\').reset_index(drop=True);x=features[[fi[i] for i in d.image_id]];y=d.domain.eq(\'OTHER\').astype(int).to_numpy();tr=d.pair_split.eq(\'train\').to_numpy();va=d.pair_split.eq(\'validation\').to_numpy();te=d.pair_split.eq(\'test\').to_numpy()\n        assert tr.any() and va.any() and te.any() and set(d.domain)=={\'ADT45\',\'OTHER\'}\n        scaler=StandardScaler().fit(x[tr]);xx=scaler.transform(x);cv=[]\n        for c in [.001,.01,.1,1.,10.]:\n            clf=LogisticRegression(C=c,max_iter=5000,solver=\'lbfgs\',random_state=19000).fit(xx[tr],y[tr])\n            if np.max(clf.n_iter_)>=5000:raise RuntimeError(\'El clasificador de dominio no convergió\')\n            cv.append((log_loss(y[va],clf.predict_proba(xx[va])),c,clf))\n        vl,c,clf=min(cv,key=lambda z:(z[0],z[1]));prob=clf.predict_proba(xx[te])[:,1];yt=y[te];td=d.loc[te].reset_index(drop=True);auc=roc_auc_score(yt,prob);ba=balanced_accuracy_score(yt,prob>=.5);rng=np.random.default_rng(19200+(protocol==\'P5\'));strata=td.groupby([\'domain\',\'label\']).indices\n        idx=np.concatenate([rng.choice(ii,size=(B,len(ii)),replace=True) for ii in strata.values()],axis=1);boot=np.array([roc_auc_score(yt[ii],prob[ii]) for ii in idx]);lo,hi=np.quantile(boot,[.025,.975])\n        rows.append(dict(protocol=protocol,n_train=int(tr.sum()),n_validation=int(va.sum()),n_test=int(te.sum()),selected_C=c,validation_nll=vl,test_auc=auc,auc_ci95_lower=lo,auc_ci95_upper=hi,test_balanced_accuracy=ba,replicates=B,scope=\'held-out separability of observed cultivar domains; not causal cultivar effect\'))\n        for j,r in td.iterrows():predrows.append(dict(protocol=protocol,image_id=r.image_id,label=r.label,domain=r.domain,prob_OTHER=prob[j]))\n        np.savez_compressed(out/(protocol+\'_domain_classifier.npz\'),coef=clf.coef_,intercept=clf.intercept_,scaler_mean=scaler.mean_,scaler_scale=scaler.scale_,C=c)\n    pd.DataFrame(rows).to_csv(out/\'19_domain_embedding_metrics.csv\',index=False);pd.DataFrame(predrows).to_csv(out/\'19_domain_embedding_predictions.csv\',index=False)\n\ndef export(out,destination):\n    import zipfile\n    out=Path(out);destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(destination,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file() and p.suffix not in [\'.pt\',\'.tmp\'] and p.resolve()!=destination.resolve():z.write(p,p.relative_to(out))\n    print(\'Resultados exportados:\',destination,\'SHA256\',digest(destination))\n\ndef audit_historical(project,inventory_path,out):\n    """Hash all available historical trained files; never infer an initial weight hash."""\n    import torch\n    folders={\'P1\':\'05_P1_30ep_noES\',\'P4\':\'06_P4_cross_cultivar_30ep_noES\',\'P5\':\'07_P5_exact_age_matched_30ep_noES\'};rows=[]\n    for r in json.loads(Path(inventory_path).read_text()):\n        p=Path(project)/\'models\'/folders[r[\'protocol\']]/r[\'run\']/\'best_model.pt\';row=dict(protocol=r[\'protocol\'],run=r[\'run\'],checkpoint_file_id=r[\'files\'][0][\'id\'],exists=p.is_file())\n        if p.is_file():\n            try:\n                from torch.torch_version import TorchVersion\n                with torch.serialization.safe_globals([TorchVersion]):c=torch.load(p,map_location=\'cpu\',weights_only=True)\n                row.update(sha256=digest(p),size=p.stat().st_size,model=c.get(\'model_key\'),seed=c.get(\'seed\'),selected_epoch=c.get(\'epoch\'),best_val_macro_f1=c.get(\'best_val_macro_f1\'),historical_initialization_hash_retained=False)\n                del c\n            except Exception as e:row.update(audit_error=type(e).__name__,audit_message=str(e)[:300])\n        rows.append(row)\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);pd.DataFrame(rows).to_csv(out/\'19_historical_checkpoint_audit.csv\',index=False)\n    print(\'Checkpoints encontrados y auditados:\',sum(r[\'exists\'] for r in rows),\'/\',len(rows))\n',encoding='utf-8')
sys.path.insert(0,str(WORK))


In [ ]:
(WORK/'paddy19_data.py').write_text('"""Paddy19.2: resolve original images, including partial caches and ZIPs.\n\nNo network or credential access occurs in this module. Kaggle restoration is a\nseparate, explicitly confirmed notebook step. Only images in frozen protocols\nare persisted; every image must match its historical SHA-256.\n"""\nfrom pathlib import Path, PurePosixPath\nimport os, json, hashlib, zipfile, shutil\nimport pandas as pd\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef required_images(meta_path,p4_path,p5_path,designs):\n    meta=pd.read_csv(meta_path)\n    p4=pd.read_csv(p4_path);p5=pd.read_csv(p5_path)\n    labels=[\'blast\',\'dead_heart\',\'downy_mildew\',\'hispa\',\'normal\']\n    ids=set(p4.loc[p4.label.isin(labels),\'image_id\'])|set(p5.image_id)\n    for d,domains in designs.values():ids.update(d.image_id)\n    selected=meta[meta.image_id.isin(ids)].copy().sort_values(\'image_id\')\n    if len(selected)!=len(ids) or selected.image_id.duplicated().any():\n        raise ValueError(\'Los IDs requeridos no coinciden con la metadata original.\')\n    return selected\n\ndef discover_sources(project,explicit=\'\',local_root=\'/content\',home=None):\n    """Bounded search; do not walk models/results or inspect credentials."""\n    project=Path(project);local_root=Path(local_root);home=Path(home) if home else Path.home()\n    roots=[];archives=[];seen=set()\n    bases=([Path(explicit).expanduser()] if explicit else [])+[\n        project/\'data\',project/\'inbox\',local_root/\'paddy-disease-classification\',\n        local_root/\'paddy15_verified_images\',local_root/\'paddy19_image_cache\',\n        home/\'.cache/kagglehub/competitions\',local_root/\'datasets\',local_root/\'data\']\n    # Other immediate runtime folders can contain an uploaded dataset ZIP.\n    if local_root.exists():\n        for p in sorted(local_root.iterdir()):\n            if p.name not in [\'drive\',\'sample_data\',\'paddy19_code\',\'paddy19_verified\']:\n                bases.append(p)\n    classes={\'blast\',\'dead_heart\',\'hispa\',\'normal\',\'downy_mildew\'}\n    for base in bases:\n        if not base.exists():continue\n        if base.is_file():\n            if zipfile.is_zipfile(base) and str(base.resolve()) not in seen:\n                archives.append(base);seen.add(str(base.resolve()))\n            continue\n        for folder,dirs,files in os.walk(base):\n            p=Path(folder);depth=len(p.relative_to(base).parts)\n            dirs[:]=sorted(d for d in dirs if d not in [\'models\',\'results\',\'exports\',\'.git\',\'config\']) if depth<5 else []\n            if p.name==\'train_images\' or classes.intersection(dirs) or (p/\'blast\').is_dir():\n                key=str(p.resolve())\n                if key not in seen:roots.append(p);seen.add(key)\n                dirs[:]=[]\n            for name in sorted(files):\n                if name.lower().endswith(\'.zip\'):\n                    z=p/name;key=str(z.resolve())\n                    if key not in seen:archives.append(z);seen.add(key)\n    return roots,archives\n\ndef restore_verified(selected,roots,archives,destination):\n    """Copy matching bytes atomically. ZIP entries are never extractall\'ed."""\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    todo={r.image_id:r for r in selected.itertuples(index=False)};receipts=[]\n    def install(iid,src=None,payload=None,origin=\'\'):\n        r=todo[iid];dst=destination/r.label/iid\n        if src is not None and sha(src)!=r.sha256:return False\n        if payload is not None and hashlib.sha256(payload).hexdigest()!=r.sha256:return False\n        dst.parent.mkdir(parents=True,exist_ok=True)\n        if src is None or Path(src).resolve()!=dst.resolve():\n            tmp=dst.with_name(dst.name+\'.restore.tmp\')\n            if src is None:tmp.write_bytes(payload)\n            else:shutil.copyfile(src,tmp)\n            tmp.replace(dst)\n        from PIL import Image\n        with Image.open(dst) as im:im.verify()\n        receipts.append(dict(image_id=iid,label=r.label,sha256=r.sha256,origin=origin))\n        del todo[iid]\n        return True\n    for root in [destination]+[Path(p) for p in roots]:\n        for iid,r in list(todo.items()):\n            for src in [root/r.label/iid,root/iid]:\n                if src.is_file() and install(iid,src=src,origin=str(src)):break\n        if not todo:break\n    for archive in archives:\n        if not todo:break\n        try:\n            with zipfile.ZipFile(archive) as z:\n                for info in z.infolist():\n                    p=PurePosixPath(info.filename.replace(\'\\\\\',\'/\'));iid=p.name\n                    if info.is_dir() or iid not in todo or \'..\' in p.parts or p.is_absolute():continue\n                    r=todo[iid]\n                    if info.file_size>50*1024*1024:continue\n                    if len(p.parts)>1 and p.parts[-2] not in [r.label,\'train_images\']:continue\n                    install(iid,payload=z.read(info),origin=str(archive)+\'::\'+info.filename)\n        except zipfile.BadZipFile:\n            print(\'ZIP incompleto, se omite:\',Path(archive).name,flush=True)\n    report=destination.parent/\'paddy19_restore_report.json\'\n    report.write_text(json.dumps(dict(required=len(selected),verified=len(selected)-len(todo),\n        missing_count=len(todo),missing_ids=sorted(todo),image_root=str(destination)),indent=2)+\'\\n\')\n    print(f\'Imágenes originales verificadas: {len(selected)-len(todo)}/{len(selected)}; faltan {len(todo)}\',flush=True)\n    return destination,sorted(todo),receipts\n',encoding='utf-8')


In [ ]:
import importlib
import paddy19_training_core as core
import paddy19_cultivar_extensions as p19
import paddy19_data as p19data
core=importlib.reload(core); p19=importlib.reload(p19); p19data=importlib.reload(p19data)
PROJECT=Path('/content/drive/MyDrive/PADDY_DOCTOR_Q1')
EXPERIMENT='19_Cultivar_Balance_LOCO_30ep_v2'
IMAGE_PATH=''  # Opcional: carpeta de imágenes, carpeta padre o ZIP original.
NUM_WORKERS=2
RUN=PROJECT/'results'/EXPERIMENT
MODELS=PROJECT/'models'/EXPERIMENT
WEIGHTS=PROJECT/'models'/'15_P5_Rematching_5x3_30ep_v1'/'pretrained_cache'
META=WORK/'inputs/nearclean.csv'
RUN.mkdir(parents=True,exist_ok=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa GPU: Entorno de ejecución > Cambiar tipo de entorno > L4 o T4. Luego ejecuta otra vez desde el inicio.')
print('GPU:',torch.cuda.get_device_name(0))
meta,designs=p19.prepare(META,RUN/'design')
for name,(d,domains) in designs.items():
    print(name,domains,len(d),'imágenes')
    display(d.groupby(['domain','pair_split','label']).size().rename('n').unstack())
print('Resultados nuevos:',RUN)


## Auditoría opcional de pesos históricos

Recorre exclusivamente las rutas de los checkpoints inventariados; guarda hashes y época. Distingue el checkpoint entrenado de la inicialización preentrenada original: la existencia del primero no recupera un hash del segundo que no fue registrado. Se verificaron previamente cuatro checkpoints representativos, uno por arquitectura, con carga estricta de sus tensores.

In [ ]:
AUDIT_HISTORICAL=False  # Tu ejecución anterior ya verificó 60/60. No hace falta repetirla.
if AUDIT_HISTORICAL:
    p19.audit_historical(PROJECT,WORK/'inputs/historical_checkpoint_inventory.json',RUN/'historical_audit')
else:
    print('Auditoría histórica omitida: ya completada en tu ejecución anterior (60/60).')


## Recuperación y verificación de imágenes

Busca carpetas anidadas, la caché del notebook 15 y ZIP locales o de `data`/`inbox` en Drive. Solo acepta imágenes que coincidan con el SHA-256 original. Si faltan, pregunta en Colab si autorizas descargarlas desde Kaggle. Solo después de tu respuesta `s` utiliza la autenticación de esa sesión o `config/kaggle.json`. La clave no se imprime ni se incluye en los resultados. No se debe introducir ninguna clave en el chat.

Guarda una copia de las imágenes requeridas en `data/raw/paddy19_verified_images` de Drive, para que no dependan de una sesión temporal. Si la cuenta de Kaggle solicita aceptar condiciones, debes hacerlo personalmente en la web de la competición. No se aceptan condiciones automáticamente.


In [ ]:
required=p19data.required_images(META,WORK/'inputs/P4.csv',WORK/'inputs/P5.csv',designs)
persistent=PROJECT/'data/raw/paddy19_verified_images'
roots,archives=p19data.discover_sources(PROJECT,IMAGE_PATH)
IMAGES,missing,receipts=p19data.restore_verified(required,roots,archives,persistent)
if missing:
    print('La sesión anterior guardó las imágenes en una caché temporal. Faltan',len(missing),'imágenes requeridas.')
    answer=input('¿Autorizas restaurarlas desde Kaggle con tu cuenta y usar config/kaggle.json si está disponible? [s/N]: ').strip().lower()
    if answer not in ('s','si','sí'):
        raise FileNotFoundError('No se autorizó la descarga. Configura IMAGE_PATH con la carpeta o el ZIP original y vuelve a ejecutar esta celda. Las imágenes ya verificadas se conservan.')
    import os,shutil
    credential=PROJECT/'config/kaggle.json'
    destination=Path.home()/'.kaggle/kaggle.json'
    if not destination.exists() and credential.is_file():
        destination.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(credential,destination);destination.chmod(0o600)
    import kagglehub
    if not destination.exists() and not os.environ.get('KAGGLE_API_TOKEN') and not (Path.home()/'.kaggle/access_token').exists():
        print('Configura tu autenticación de Kaggle en el diálogo privado de Colab.')
        kagglehub.login()
    try:
        downloaded=Path(kagglehub.competition_download('paddy-disease-classification'))
    except Exception as e:
        raise RuntimeError('Kaggle no entregó los datos. Revisa la autenticación y acepta personalmente las condiciones de paddy-disease-classification en Kaggle; después repite esta celda. Error: '+type(e).__name__) from None
    roots,archives=p19data.discover_sources(PROJECT,str(downloaded))
    IMAGES,missing,receipts=p19data.restore_verified(required,roots,archives,persistent)
    if missing:
        raise FileNotFoundError(f'Persisten {len(missing)} imágenes ausentes o distintas del SHA-256 original. Consulta data/raw/paddy19_restore_report.json. No se cambia el protocolo para ignorarlas.')
print('Listo. Imágenes originales disponibles para ambos análisis:',len(required))
print('Carpeta persistente:',IMAGES)


## Distancia visual mediante clasificación de dominio

La representación ResNet-50 no se ajusta al arroz. Se conserva el tag, el hash de los pesos y los identificadores de cada vector. No se utiliza el test para elegir C ni el extractor. La comparación P4/P5 mantiene las cinco clases, pero cambia muestras y tamaños; por ello una diferencia entre AUC no aísla un efecto de edad.

In [ ]:
try:
    p19.domain_embeddings(META,WORK/'inputs/P4.csv',WORK/'inputs/P5.csv',IMAGES,RUN/'embeddings',WEIGHTS,workers=NUM_WORKERS,B=5000)
finally:
    p19.export(RUN,PROJECT/'exports'/'19_PaddyDoctor_RESULTADOS_PARCIAL.zip')
display(__import__('pandas').read_csv(RUN/'embeddings/19_domain_embedding_metrics.csv'))


## 48 ajustes nuevos y evaluación pareada

Los archivos se guardan en Drive al terminar cada época y corrida. Las corridas completas se verifican y omiten al reanudar; una corrida interrumpida reinicia desde los pesos preentrenados. No se mezclan configuraciones o entornos distintos dentro de una misma carpeta. No existe reanudación exacta de la época incompleta.

La selección del checkpoint y la temperatura se ajustan en validación fuente. Los intervalos finales remuestrean imágenes por clase y mantienen las tres semillas como realizaciones fijas. No cuentan las semillas como nuevas imágenes.

In [ ]:
try:
    p19.run_training(META,RUN,IMAGES,MODELS,WEIGHTS,workers=NUM_WORKERS)
except BaseException:
    p19.export(RUN,PROJECT/'exports'/'19_PaddyDoctor_RESULTADOS_PARCIAL.zip')
    raise


## Exportación para incorporar resultados al manuscrito

Debe haber 24 corridas completas por diseño, 48 en total. El ZIP incluye asignaciones, predicciones, logits, historiales, intervalos, metadatos y comprobaciones; los checkpoints grandes permanecen en `models`. Entrega `19_PaddyDoctor_RESULTADOS.zip` para integrar los resultados reales. No se declararán estos comentarios resueltos solo por disponer del notebook.

In [ ]:
import pandas as pd
summaries=[json.loads((RUN/protocol/'completion_summary.json').read_text()) for protocol in ['P6','P7']]
if not all(s['complete'] and s['completed_runs']==24 for s in summaries):
    raise RuntimeError('Faltan corridas. Reanuda la celda de entrenamiento; no se exporta como completo.')
required_results=[RUN/'19_paired_image_intervals.csv',RUN/'19_losses_by_seed.csv',RUN/'embeddings/19_domain_embedding_metrics.csv']
if not all(p.is_file() for p in required_results):
    raise RuntimeError('Falta un análisis final. Ejecuta embeddings y entrenamiento antes de exportar.')
display(pd.read_csv(RUN/'19_paired_image_intervals.csv'))
p19.export(RUN,PROJECT/'exports'/'19_PaddyDoctor_RESULTADOS.zip')
print('COMPLETADO: 48 entrenamientos y comparación con embeddings. Este es el ZIP que debes devolver.')
